# Lesson 1 - Exercise - Part 2

Solutions des 4 exercices :

1. Suite de Fibonacci stockee dans un dictionnaire
2. Compter les occurrences des mots d'une phrase
3. Inverser un dictionnaire (les cles deviennent les valeurs)
4. Dictionnaires imbriques : moyennes d'une ecole

---
## Exercice 1 - Suite de Fibonacci

Calculer la suite de Fibonacci et stocker les resultats dans un dictionnaire,
sous la forme `{indice: valeur}`.

In [ ]:
def fibonacci(n):
    """Retourne {0: F0, 1: F1, ..., n: Fn} avec F0 = 0 et F1 = 1."""
    if n < 0:
        raise ValueError("n doit etre positif ou nul")

    result = {0: 0}
    if n == 0:
        return result

    result[1] = 1
    for i in range(2, n + 1):
        result[i] = result[i - 1] + result[i - 2]
    return result


fib = fibonacci(10)
print("Dictionnaire :", fib)
print("F(10)        :", fib[10])
print("Cas limites  :", fibonacci(0), fibonacci(1))

for indice, valeur in fib.items():
    print(f"F({indice:>2}) = {valeur}")

Dictionnaire : {0: 0, 1: 1, 2: 1, 3: 2, 4: 3, 5: 5, 6: 8, 7: 13, 8: 21, 9: 34, 10: 55}
F(10)        : 55
Cas limites  : {0: 0} {0: 0, 1: 1}
F( 0) = 0
F( 1) = 1
F( 2) = 1
F( 3) = 2
F( 4) = 3
F( 5) = 5
F( 6) = 8
F( 7) = 13
F( 8) = 21
F( 9) = 34
F(10) = 55


### Variante : memoisation

On peut aussi remplir le dictionnaire de facon recursive. Le dictionnaire sert
alors de **cache** : chaque valeur n'est calculee qu'une seule fois, ce qui evite
l'explosion du nombre d'appels de la version recursive naive.

In [2]:
def fibonacci_memo(n, cache):
    """Version recursive. Le dictionnaire cache sert de memoire."""
    if n in cache:
        return cache[n]

    cache[n] = fibonacci_memo(n - 1, cache) + fibonacci_memo(n - 2, cache)
    return cache[n]


cache = {0: 0, 1: 1}
fibonacci_memo(30, cache)

print("F(30) :", cache[30])
print("Nombre de valeurs en cache :", len(cache))
print("Meme resultat que la version iterative :", fibonacci(30) == cache)

F(30) : 832040
Nombre de valeurs en cache : 31
Meme resultat que la version iterative : True


---
## Exercice 2 - Compter les occurrences

Creer un dictionnaire dont les cles sont les mots d'une phrase et les valeurs
leur nombre d'occurrences.

Exemple : `"hello world hello"` donne `{'hello': 2, 'world': 1}`

In [3]:
def count_words(sentence):
    """Compte les occurrences de chaque mot d'une phrase."""
    counts = {}
    for word in sentence.split():
        if word in counts:
            counts[word] += 1
        else:
            counts[word] = 1
    return counts


# Tests
print(count_words("hello world hello"))
print(count_words("le chat dort le chien dort aussi"))
print(count_words(""))

{'hello': 2, 'world': 1}
{'le': 2, 'chat': 1, 'dort': 2, 'chien': 1, 'aussi': 1}
{}


### Variante 1 : `dict.get`

`counts.get(word, 0)` retourne 0 quand le mot n'est pas encore dans le
dictionnaire, ce qui supprime le `if / else`.

In [ ]:
def count_words_get(sentence):
    """Meme chose, en une seule ligne dans la boucle grace a dict.get."""
    counts = {}
    for word in sentence.split():
        counts[word] = counts.get(word, 0) + 1
    return counts


print(count_words_get("hello world hello"))

### Variante 2 : nettoyage du texte et `Counter`

En pratique il faut souvent normaliser le texte avant de compter : passer en
minuscules et retirer la ponctuation, sinon `"Hello"`, `"hello"` et `"hello,"`
comptent pour trois mots differents.

`collections.Counter` fait ensuite le comptage pour nous.

In [ ]:
from collections import Counter
import string


def count_words_clean(sentence):
    """Compte les mots en ignorant la casse et la ponctuation."""
    cleaned = sentence.lower()
    for char in string.punctuation:
        cleaned = cleaned.replace(char, " ")
    return dict(Counter(cleaned.split()))


phrase = "Hello world, hello Python. HELLO again!"

print("Sans nettoyage :", count_words(phrase))
print("Avec nettoyage :", count_words_clean(phrase))

# Les mots les plus frequents
compteur = Counter(count_words_clean(phrase))
print("Top 2          :", compteur.most_common(2))

---
## Exercice 3 - Inverser un dictionnaire

Inverser un dictionnaire : les cles deviennent les valeurs et les valeurs
deviennent les cles.

*Remarque :* les valeurs du dictionnaire d'origine sont uniques.

Exemple : `{"a": 1, "b": 2}` devient `{1: 'a', 2: 'b'}`

In [1]:
def invert_dict(d):
    """Inverse un dictionnaire dont les valeurs sont uniques."""
    return {value: key for key, value in d.items()}


# Tests
original = {"a": 1, "b": 2, "c": 3}
inverse = invert_dict(original)

print("Original :", original)
print("Inverse  :", inverse)
print("Double inversion == original :", invert_dict(inverse) == original)

Original : {'a': 1, 'b': 2, 'c': 3}
Inverse  : {1: 'a', 2: 'b', 3: 'c'}
Double inversion == original : True


### Version avec une boucle

La comprehension de dictionnaire ci-dessus est equivalente a cette boucle
explicite.

In [2]:
def invert_dict_loop(d):
    """Meme resultat, ecrit avec une boucle."""
    result = {}
    for key, value in d.items():
        result[value] = key
    return result


print(invert_dict_loop({"a": 1, "b": 2}))

{1: 'a', 2: 'b'}


### Variante : et si les valeurs ne sont pas uniques ?

L'enonce garantit des valeurs uniques. Sinon la derniere paire rencontree ecrase
les precedentes et on perd de l'information. Pour ne rien perdre, on regroupe les
cles d'origine dans une liste.

Attention aussi : une valeur ne peut devenir une cle que si elle est
**hachable**. Une liste, par exemple, ne peut pas servir de cle.

In [ ]:
# Probleme : deux cles partagent la meme valeur
doublons = {"a": 1, "b": 2, "c": 1}
print("Inversion naive (on perd 'a') :", invert_dict(doublons))


def invert_dict_multi(d):
    """Inverse un dictionnaire en regroupant les cles qui partagent une valeur."""
    result = {}
    for key, value in d.items():
        result.setdefault(value, []).append(key)
    return result


print("Inversion groupee             :", invert_dict_multi(doublons))

---
## Exercice 4 - Dictionnaires imbriques

Creer un dictionnaire representant une ecole avec des classes, chaque classe
contenant des eleves (sous forme de dictionnaires).

Puis calculer la moyenne de chaque eleve, de chaque classe, et de l'ecole entiere.

In [ ]:
school = {
    "Class A": {
        "Alice": {"age": 12, "grades": [15, 18, 14]},
        "Bob": {"age": 13, "grades": [12, 16, 17]},
    },
    "Class B": {
        "Charlie": {"age": 12, "grades": [19, 20, 18]},
        "Diana": {"age": 13, "grades": [11, 13, 10]},
    },
}

# Un apercu de la structure
for class_name, students in school.items():
    print(class_name)
    for student_name, info in students.items():
        print(f"  {student_name:<8} age={info['age']}  notes={info['grades']}")

### Moyenne de chaque eleve

`sum(grades) / len(grades)` donne la moyenne. On protege le cas d'un eleve sans
note, pour eviter une division par zero.

In [ ]:
def average(numbers):
    """Moyenne d'une liste de nombres. Retourne None si la liste est vide."""
    if not numbers:
        return None
    return sum(numbers) / len(numbers)


def student_averages(school):
    """Retourne {classe: {eleve: moyenne}}."""
    return {
        class_name: {
            student_name: average(info["grades"])
            for student_name, info in students.items()
        }
        for class_name, students in school.items()
    }


moyennes_eleves = student_averages(school)

for class_name, students in moyennes_eleves.items():
    print(class_name)
    for student_name, moyenne in students.items():
        print(f"  {student_name:<8} {moyenne:.2f}")

### Moyenne de chaque classe et de l'ecole

Attention : la moyenne d'une classe est la moyenne de **toutes les notes** de la
classe, et non la moyenne des moyennes des eleves. Les deux ne coincident que si
tous les eleves ont le meme nombre de notes.

On met donc toutes les notes a plat avant de calculer.

In [ ]:
def class_grades(students):
    """Toutes les notes d'une classe, rassemblees dans une seule liste."""
    grades = []
    for info in students.values():
        grades.extend(info["grades"])
    return grades


def class_averages(school):
    """Retourne {classe: moyenne de la classe}."""
    return {
        class_name: average(class_grades(students))
        for class_name, students in school.items()
    }


def school_average(school):
    """Moyenne de toutes les notes de l'ecole."""
    all_grades = []
    for students in school.values():
        all_grades.extend(class_grades(students))
    return average(all_grades)


moyennes_classes = class_averages(school)

for class_name, moyenne in moyennes_classes.items():
    print(f"{class_name} : {moyenne:.2f}")

print(f"Ecole   : {school_average(school):.2f}")

### Bilan complet

Un seul affichage qui reprend les trois niveaux : eleve, classe, ecole.

In [ ]:
def report(school):
    """Affiche les moyennes par eleve, par classe, puis pour l'ecole."""
    eleves = student_averages(school)
    classes = class_averages(school)

    for class_name, students in eleves.items():
        print(f"{class_name} (moyenne {classes[class_name]:.2f})")
        for student_name, moyenne in students.items():
            print(f"    {student_name:<8} {moyenne:.2f}")

    print(f"\nMoyenne de l'ecole : {school_average(school):.2f}")


report(school)

### Cas limites

Une classe vide ou un eleve sans note ne doivent pas faire planter le calcul :
`average` retourne alors `None`.

In [ ]:
ecole_vide = {
    "Class A": {"Eve": {"age": 12, "grades": []}},
    "Class B": {},
}

print("Moyennes eleves  :", student_averages(ecole_vide))
print("Moyennes classes :", class_averages(ecole_vide))
print("Moyenne ecole    :", school_average(ecole_vide))

---
## Recapitulatif

Un test global qui rejoue les 4 exercices.

In [ ]:
print("1. fibonacci(8)                     ->", fibonacci(8))
print("2. count_words('hello world hello') ->", count_words("hello world hello"))
print("3. invert_dict({'a': 1, 'b': 2})    ->", invert_dict({"a": 1, "b": 2}))
print("4. class_averages(school)           ->", class_averages(school))
print("4. school_average(school)           ->", round(school_average(school), 2))